In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.preprocessing import StandardScaler


In [2]:
data_dir = Path(r"C:\Users\Richa\OneDrive\文档\idx exchange")
data_dir


WindowsPath('C:/Users/Richa/OneDrive/文档/idx exchange')

Load and Combine Multiple Months of Data



In [3]:
files = [
    "CRMLSSold202510.csv",
    "CRMLSSold202511.csv",
    "CRMLSSold202512.csv",
    "CRMLSSold202601.csv",
    "CRMLSSold202602.csv",
    "CRMLSSold202603.csv",
]

dfs = []
for f in files:
    path = data_dir / f
    temp = pd.read_csv(path)
    temp["SourceFile"] = f
    dfs.append(temp)

df = pd.concat(dfs, ignore_index=True)

print(df.shape)
df.head(5)

C:\Users\Richa\AppData\Local\Temp\ipykernel_45516\1927989732.py:13: DtypeWarning: Columns (0: WaterfrontYN, 1: PostalCode) have mixed types. Specify dtype option on import or set low_memory=False.
  temp = pd.read_csv(path)


(120053, 79)


,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,...,LotSizeArea,MainLevelBedrooms,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,SourceFile
0,NorthSanLuisObispo,NorthSanLuisObispo,NaN,NaN,NaN,NaN,NaN,NaN,479293596,newlin@pacificaCRE.com,...,1807.0,NaN,NaN,NaN,NaN,93446,NaN,1807.0,NaN,CRMLSSold202510.csv
1,TheInlandGateway,TheInlandGateway,NaN,True,NaN,NaN,NaN,12000.0,446914808,nleimers@hotmail.com,...,14253.0,NaN,False,NaN,NaN,92325,0.0,14253.0,NaN,CRMLSSold202510.csv
2,SanDiego,SanDiego,Wood,True,NaN,NaN,False,1695.0,445176588,mannybehar@yahoo.com,...,NaN,NaN,NaN,0.0,NaN,92126,0.0,NaN,NaN,CRMLSSold202510.csv
3,SanDiego,SanDiego,NaN,False,NaN,NaN,False,950000.0,1145282039,chase@cromwellhomegroup.com,...,NaN,NaN,False,2.0,NaN,91901,NaN,NaN,NaN,CRMLSSold202510.csv
4,SanDiego,SanDiego,NaN,False,NaN,NaN,False,970000.0,1145278097,rory@corneliusestates.net,...,NaN,NaN,False,2.0,NaN,92115,NaN,NaN,NaN,CRMLSSold202510.csv


In [4]:
##Filter Residential Single-Family Properties
required_cols = ["PropertyType", "PropertySubType", "ClosePrice", "SourceFile"]
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise KeyError(f"Missing required columns: {missing_required}")

df = df[
    (df["PropertyType"] == "Residential") &
    (df["PropertySubType"] == "SingleFamilyResidence")
].copy()

print("Filtered shape:", df.shape)
df.head()


Filtered shape: (59440, 79)


,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,...,LotSizeArea,MainLevelBedrooms,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,SourceFile
3,SanDiego,SanDiego,NaN,False,NaN,NaN,False,950000.0,1145282039,chase@cromwellhomegroup.com,...,NaN,NaN,False,2.0,NaN,91901,NaN,NaN,NaN,CRMLSSold202510.csv
4,SanDiego,SanDiego,NaN,False,NaN,NaN,False,970000.0,1145278097,rory@corneliusestates.net,...,NaN,NaN,False,2.0,NaN,92115,NaN,NaN,NaN,CRMLSSold202510.csv
5,SanDiego,SanDiego,NaN,False,NaN,NaN,False,575000.0,1145277224,garylawrence@kw.com,...,NaN,NaN,False,2.0,NaN,91945,NaN,NaN,NaN,CRMLSSold202510.csv
7,BeverlyHillsGreaterLa,BeverlyHillsGreaterLa,Tile,True,NaN,NaN,False,1243810.0,1145272633,taylor@acme-re.com,...,2066.0,NaN,True,NaN,NaN,90065,141.25,2066.0,NaN,CRMLSSold202510.csv
8,BeverlyHillsGreaterLa,BeverlyHillsGreaterLa,Tile,True,NaN,NaN,False,1227060.0,1145272243,taylor@acme-re.com,...,1804.0,NaN,True,NaN,NaN,90065,141.25,1804.0,NaN,CRMLSSold202510.csv


In [5]:
#remove duplicates
print("Duplicate rows before:", df.duplicated().sum())
df = df.drop_duplicates().copy()
print("Duplicate rows after:", df.duplicated().sum())


Duplicate rows before: 2
Duplicate rows after: 0


In [6]:
##Create Train and Test Splits by Month
latest_file = sorted(df["SourceFile"].unique())[-1]

train_df = df[df["SourceFile"] != latest_file].copy()
test_df = df[df["SourceFile"] == latest_file].copy()

print("Latest month used for test:", latest_file)
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)


Latest month used for test: CRMLSSold202603.csv
Train shape: (48262, 79)
Test shape: (11176, 79)


In [7]:
##Select Modeling Columns
target_col = "ClosePrice"

numeric_candidates = [
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "YearBuilt",
    "GarageSpaces",
    "Latitude",
    "Longitude",
    "CumulativeDaysOnMarket",
    "DaysOnMarket",
]

categorical_candidates = [
    "City",
    "CountyOrParish",
    "PostalCode",
    "StateOrProvince",
    "Heating",
    "Cooling",
    "GarageYN",
    "FireplaceYN",
    "PoolYN",
]

numeric_cols = [c for c in numeric_candidates if c in train_df.columns]
cat_cols = [c for c in categorical_candidates if c in train_df.columns]

print("Numeric columns:", numeric_cols)
print("Categorical columns before filtering:", cat_cols)


Numeric columns: ['LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 'LotSizeSquareFeet', 'YearBuilt', 'GarageSpaces', 'Latitude', 'Longitude', 'DaysOnMarket']
Categorical columns before filtering: ['City', 'CountyOrParish', 'PostalCode', 'StateOrProvince', 'FireplaceYN']


In [8]:
#missing value handling
train_df = train_df.dropna(subset=[target_col]).copy()
test_df = test_df.dropna(subset=[target_col]).copy()

for col in numeric_cols:
    median_value = train_df[col].median()
    train_df[col] = train_df[col].fillna(median_value)
    test_df[col] = test_df[col].fillna(median_value)

for col in cat_cols:
    train_df[col] = train_df[col].fillna("Unknown").astype(str)
    test_df[col] = test_df[col].fillna("Unknown").astype(str)

print("Missing values left in train:", int(train_df.isnull().sum().sum()))
print("Missing values left in test:", int(test_df.isnull().sum().sum()))


Missing values left in train: 1112850
Missing values left in test: 259271


In [9]:
"""Remove High-Cardinality Categorical Columns

Very high-cardinality columns can create too many dummy variables.
We keep only categorical columns with a manageable number of unique values in the training set."""
max_unique_allowed = 30
kept_cat_cols = [
    col for col in cat_cols
    if train_df[col].nunique(dropna=False) <= max_unique_allowed
]

dropped_cat_cols = [col for col in cat_cols if col not in kept_cat_cols]

print("Kept categorical columns:", kept_cat_cols)
print("Dropped categorical columns:", dropped_cat_cols)


Kept categorical columns: ['StateOrProvince', 'FireplaceYN']
Dropped categorical columns: ['City', 'CountyOrParish', 'PostalCode']


In [10]:
# categorical encoding to numerical values
train_encoded = pd.get_dummies(train_df, columns=kept_cat_cols, drop_first=True)
test_encoded = pd.get_dummies(test_df, columns=kept_cat_cols, drop_first=True)

train_encoded, test_encoded = train_encoded.align(test_encoded, join="left", axis=1, fill_value=0)

print("Encoded train shape:", train_encoded.shape)
print("Encoded test shape:", test_encoded.shape)


Encoded train shape: (48262, 80)
Encoded test shape: (11176, 80)


standardize numeric features

numeric features are scaled using statistics from the training set only.
The target variable is left unchanged


In [11]:
scaler = StandardScaler()

numeric_cols_to_scale = [c for c in numeric_cols if c in train_encoded.columns]

train_encoded[numeric_cols_to_scale] = scaler.fit_transform(train_encoded[numeric_cols_to_scale])
test_encoded[numeric_cols_to_scale] = scaler.transform(test_encoded[numeric_cols_to_scale])

numeric_cols_to_scale


['LivingArea',
 'BedroomsTotal',
 'BathroomsTotalInteger',
 'LotSizeSquareFeet',
 'YearBuilt',
 'GarageSpaces',
 'Latitude',
 'Longitude',
 'DaysOnMarket']

final clean up



In [12]:
drop_cols = ["PropertyType", "PropertySubType", "SourceFile"]

train_final = train_encoded.drop(columns=[c for c in drop_cols if c in train_encoded.columns]).copy()
test_final = test_encoded.drop(columns=[c for c in drop_cols if c in test_encoded.columns]).copy()

print("Final train shape:", train_final.shape)
print("Final test shape:", test_final.shape)


Final train shape: (48262, 77)
Final test shape: (11176, 77)


saving data set





In [13]:
train_output = data_dir / "cleaned_train.csv"
test_output = data_dir / "cleaned_test.csv"

train_final.to_csv(train_output, index=False)
test_final.to_csv(test_output, index=False)

print("Saved:", train_output)
print("Saved:", test_output)


Saved: C:\Users\Richa\OneDrive\文档\idx exchange\cleaned_train.csv
Saved: C:\Users\Richa\OneDrive\文档\idx exchange\cleaned_test.csv



Preprocessing Summary

Combined multiple months of CRMLSSold data.
Filtered the dataset to residential single-family properties.
Removed duplicate rows.
Split the data chronologically using the most recent month as the test set.
Imputed missing numeric values with the training median.
Filled missing categorical values with `Unknown`.
One-hot encoded categorical variables with manageable cardinality.
Standardized numeric variables.
Exported cleaned training and test CSV files.
